# Data prep 

Prepping background data for LCA tool. 
- `../data/processed/unit_burdens.csv` - unit EF burdens (e.g. kgCO2) of 
    - material production and transportation to factory for multiple biobased materials (kgCO2/kg)
    - machine wear, based on ecoinvent dataset for "market for industrial machine" (kgCO2/kg)
    - energy use, based on econinvent dataset for "market for electricity", for different EU countries (kWh/kg)
- `../data/processed/unit_benefits.csv` - unit benefits of multiple materials, including information on 
    - carbon content (kgCO2)
    - rotation period (yrs)

## Notebook setup 

### Brightway databse setup 

#### Setup and version pinning

In [1]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
import importlib.metadata

# ── Version pinning ───────────────────────────────────────────────────────────
for pkg in ["bw2data", "bw2io", "bw2calc"]:
    print(f"  {pkg}: {importlib.metadata.version(pkg)}")
print()

# ── Project and database naming ───────────────────────────────────────────────
PROJECT_NAME     = "biobased_construction_impact_calculator"
EI_VERSION       = "3.12"
EI_MODEL         = "cutoff"
EI_DB_NAME       = f"ecoinvent-{EI_VERSION}-{EI_MODEL}"       # "ecoinvent-3.12-cutoff"
BIOSPHERE_NAME   = f"ecoinvent-{EI_VERSION}-biosphere"        # "ecoinvent-3.12-biosphere"

# ── LCIA method naming ────────────────────────────────────────────────────────
# This ecoinvent version namespaces methods one level deeper than older
# Brightway convention: (namespace, family, category, indicator) instead of
# (family, category, indicator).
METHOD_NAMESPACE = f"ecoinvent-{EI_VERSION}"                  # "ecoinvent-3.12"
METHOD_FAMILY    = "EF v3.1"                                  # not "EF v3.1 no LT"

print(f"Target project:    {PROJECT_NAME}")
print(f"Target database:   {EI_DB_NAME}")
print(f"Target biosphere:  {BIOSPHERE_NAME}")
print(f"Method namespace:  {METHOD_NAMESPACE}")
print(f"Method family:     {METHOD_FAMILY}")

  bw2data: 4.4.2
  bw2io: 0.9.6
  bw2calc: 2.0.1

Target project:    biobased_construction_impact_calculator
Target database:   ecoinvent-3.12-cutoff
Target biosphere:  ecoinvent-3.12-biosphere
Method namespace:  ecoinvent-3.12
Method family:     EF v3.1


#### Create project and import ecoinvent

In [2]:
import os

# ── Connect to (or create) the project ────────────────────────────────────────
bd.projects.set_current(PROJECT_NAME)
print(f"Active project: {bd.projects.current}")

# ── Import ecoinvent, if not already present ──────────────────────────────────
if EI_DB_NAME in bd.databases:
    print(f"'{EI_DB_NAME}' already imported ({len(bd.Database(EI_DB_NAME))} datasets) — skipping.")
else:
    ei_username = os.environ.get("ECOINVENT_USERNAME")
    ei_password = os.environ.get("ECOINVENT_PASSWORD")

    bi.import_ecoinvent_release(
        version=EI_VERSION,
        system_model=EI_MODEL,
        username=ei_username,
        password=ei_password,
    )
    print(f"Imported '{EI_DB_NAME}': {len(bd.Database(EI_DB_NAME))} datasets")

ei = bd.Database(EI_DB_NAME)

Active project: biobased_construction_impact_calculator
'ecoinvent-3.12-cutoff' already imported (26533 datasets) — skipping.


#### Define EF v3.1 impact categories

In [3]:
# ── EF v3.1 methods, namespaced under this ecoinvent version ─────────────────
EF_METHODS = [
    m for m in bd.methods
    if m[0] == METHOD_NAMESPACE and m[1] == METHOD_FAMILY
]
print(f"Found {len(EF_METHODS)} EF v3.1 method tuples")

# ── Units lookup, keyed on method[2] (the category level) ────────────────────
EF_UNITS = {
    'acidification':                                     'mol H+-eq',
    'climate change':                                     'kg CO2-eq',
    'climate change: biogenic':                           'kg CO2-eq',
    'climate change: fossil':                             'kg CO2-eq',
    'climate change: land use and land use change':       'kg CO2-eq',
    'ecotoxicity: freshwater':                            'CTUe',
    'ecotoxicity: freshwater, inorganics':                'CTUe',
    'ecotoxicity: freshwater, organics':                  'CTUe',
    'energy resources: non-renewable':                    'MJ',
    'eutrophication: freshwater':                         'kg P-eq',
    'eutrophication: marine':                             'kg N-eq',
    'eutrophication: terrestrial':                        'mol N-eq',
    'human toxicity: carcinogenic':                       'CTUh',
    'human toxicity: carcinogenic, inorganics':           'CTUh',
    'human toxicity: carcinogenic, organics':             'CTUh',
    'human toxicity: non-carcinogenic':                   'CTUh',
    'human toxicity: non-carcinogenic, inorganics':       'CTUh',
    'human toxicity: non-carcinogenic, organics':         'CTUh',
    'ionising radiation: human health':                   'kBq U235-eq',
    'land use':                                           'dimensionless (soil quality index)',
    'material resources: metals/minerals':                'kg Sb-eq',
    'ozone depletion':                                    'kg CFC-11-eq',
    'particulate matter formation':                       'disease incidence',
    'photochemical oxidant formation: human health':      'dimensionless (tropospheric ozone concentration increase)',
    'water use':                                           'm3 world-eq deprived',
}

missing_units = [m[2] for m in EF_METHODS if m[2] not in EF_UNITS]
if missing_units:
    print(f"⚠ {len(missing_units)} categories have no unit defined: {missing_units}")
else:
    print(f"All {len(EF_METHODS)} categories have a unit defined.")

Found 25 EF v3.1 method tuples
All 25 categories have a unit defined.


### Helper functions 

e.g. ecoinvent database lookup 

In [4]:
def find_ei(name, location, ref_product=None):
    """Look up a single ecoinvent activity by exact name and location."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Ecoinvent dataset not found: '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} — using first")
    return results[0]


def search_ei(keyword, max_results=20):
    """
    Broad keyword search across ecoinvent. Used to verify dataset names
    before hard-coding them in find_ei().
    """
    keyword_lower = keyword.lower()
    results = [a for a in ei if keyword_lower in a['name'].lower()]
    print(f"Found {len(results)} dataset(s) matching '{keyword}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print(f"  unit:     {a.get('unit', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more, not shown")

def get_kg_conversion_factor(act):
    """kg per 1 reference unit — 1.0 for kg-based activities, else derived from the 'wet mass' property."""
    if (act.get("unit") or "").lower() == "kilogram":
        return 1.0
    for exc in act.production():
        wet_mass = exc.get("properties", {}).get("wet mass", {}).get("amount")
        if wet_mass:
            return wet_mass
    return None

## Unit burdens

#### Load unit burden data sources (from Google Sheets)

In [14]:
import pandas as pd
import re

SHEET_ID = "1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4"
GID = "1997851380"  # unit_burdens_dataSources tab

sheet_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
df_sources = pd.read_csv(sheet_url, na_values=["n/a"])

# section-header rows (e.g. "Bio-based materials") have no `unit` value — drop them
df_sources = df_sources[df_sources["unit"].notna()].copy()

for col in ["activity_name", "unit", "material_source", "ecoinventDataset_name",
            "geographicalCoverage", "referenceProduct", "alternative_data_source"]:
    df_sources[col] = df_sources[col].astype("string").str.strip()

def normalize_unit(raw):
    """'1 kg' -> 'kg', '1 ton km' -> 'ton km', '1 kwh' -> 'kwh'."""
    return re.sub(r"^1\s*", "", str(raw)).strip()

df_sources["material_unit"] = df_sources["unit"].apply(normalize_unit)

print(f"Loaded {len(df_sources)} activities from '{sheet_url.split('gid=')[0]}...'")
df_sources.head()

Loaded 32 activities from 'https://docs.google.com/spreadsheets/d/1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4/export?format=csv&...'


,activity_name,unit,material_source,ecoinventDataset_name,geographicalCoverage,referenceProduct,alternative_data_source,comments,material_unit
1,Peas,1 kg,virgin,market for protein pea,GLO,protein pea,<NA>,NaN,kg
2,Cotton fiber,1 kg,virgin,market for seed-cotton,GLO,seed-cotton,<NA>,NaN,kg
3,Hemp fiber,1 kg,virgin,"decorticated fibre production, hemp",FR,"decorticated fibre, hemp",<NA>,NaN,kg
4,Hard wood,1 kg,virgin,"market for sawnwood, hardwood, raw",GLO,"sawnwood, hardwood, raw",<NA>,NaN,kg
5,Soft wood,1 kg,virgin,"market for sawnwood, softwood, raw",GLO,"sawnwood, softwood, raw",<NA>,NaN,kg


#### Pea protein binder — precomputed in biopol_lca

In [6]:
CATEGORY_RENAME_MAP = {"photochemical ozone formation": "photochemical oxidant formation: human health"}

bd.projects.set_current("biopol_lca")
binder_act = bd.get_node(name="pea protein binder production", database="lca_database_3DPrintedBiopol")

binder_scores = {}
for method in [m for m in bd.methods if m[0] == "EF v3.1"]:
    lca = bc.LCA({binder_act: 1}, method)
    lca.lci()
    lca.lcia()
    binder_scores[CATEGORY_RENAME_MAP.get(method[1], method[1])] = lca.score

bd.projects.set_current(PROJECT_NAME)  # switch back before continuing

print(f"Computed {len(binder_scores)} category scores for the pea protein binder")

Computed 25 category scores for the pea protein binder


#### Resolve the "Road transport" reference activity

Used for every row whose `alternative_data_source` is `"road transport 50 km"` — scaled to a 1 kg / 50 km basis.

In [7]:
transport_row = df_sources[df_sources["activity_name"] == "Road transport"].iloc[0]

transport_act = find_ei(
    name=transport_row["ecoinventDataset_name"],
    location=transport_row["geographicalCoverage"],
    ref_product=transport_row["referenceProduct"] if pd.notna(transport_row["referenceProduct"]) else None,
)

TRANSPORT_DISTANCE_KM = 50  # generic collection distance assumption
tkm_per_kg = TRANSPORT_DISTANCE_KM * 0.001  # t·km per kg material

print(f"Resolved transport activity: '{transport_act['name']}' | {transport_act['location']}")

Resolved transport activity: 'market for transport, freight, lorry, unspecified' | RER


#### Get European location codes from ecoinvent

Instead of hardcoding country codes, pull them from the technosphere inputs of ecoinvent's
`"market group for electricity, low voltage"` (RER) — this is itself an aggregation of the
country-level electricity markets, so its inputs give us the definitive list of countries
that `"lookup: all european countries"` should resolve to.

In [8]:
def get_locations_from_market_group(market_group_name, market_group_location, constituent_name):
    """
    Ecoinvent's regional market groups aggregate country-level markets as technosphere
    inputs. Return the location codes of those inputs, rather than hardcoding a list.
    """
    market_group_act = find_ei(market_group_name, market_group_location)
    locations = sorted({
        exc.input["location"]
        for exc in market_group_act.technosphere()
        if exc.input.get("name") == constituent_name
    })
    return locations

EUROPEAN_LOCATIONS = get_locations_from_market_group(
    market_group_name="market group for electricity, low voltage",
    market_group_location="RER",
    constituent_name="market for electricity, low voltage",
)
EUROPEAN_LOCATIONS = EUROPEAN_LOCATIONS + get_locations_from_market_group(
    market_group_name="market group for electricity, low voltage",
    market_group_location="Europe without Switzerland",
    constituent_name="market for electricity, low voltage",
)

print(f"Found {len(EUROPEAN_LOCATIONS)} European country markets: {EUROPEAN_LOCATIONS}")

Found 40 European country markets: ['CH', 'AL', 'AT', 'BA', 'BE', 'BG', 'BY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GI', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'MD', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UA', 'XK']


#### Systematic burden extraction (with caching)

Skips any `(material_name, material_source)` combination already present in an existing
`unit_burdens.csv`, so re-running the notebook only fills in what's missing. Set
`FORCE_REFRESH` to a list of `material_name` values to force those specific activities
to be re-downloaded regardless of cache.

Branching logic per row in `df_sources`:
- if `ecoinventDataset_name` is set:
    - if `geographicalCoverage == "lookup: all european countries"` → score for every country in `EUROPEAN_LOCATIONS`
    - else → single lookup at the given location
- elif `alternative_data_source == "road transport 50 km"` → use `transport_act`, scaled by `tkm_per_kg`
- elif `alternative_data_source == "biopol_lca_database"` → pull from `binder_scores`

In [17]:
OUTPUT_DIR = "../../data/processed"  # adjust to match your repo structure
OUTPUT_PATH = f"{OUTPUT_DIR}/unit_burdens.csv"

FORCE_REFRESH = []  # e.g. ["Hard wood"] to force re-download of specific activities
LOOKUP_SECONDS_ESTIMATE = 20  # observed average time per unique ecoinvent lookup

EXPECTED_N_CATEGORIES = len({m[2] for m in EF_METHODS})


# ── Cache state (disk) ─────────────────────────────────────────────────────────
def load_cache_state():
    """Read unit_burdens.csv (if present) and split it into complete vs. incomplete
    (material_name, material_source) activities. Complete ones are skipped this run."""
    if not os.path.exists(OUTPUT_PATH):
        print(f"No existing {OUTPUT_PATH} found — downloading everything")
        return pd.DataFrame(), set()

    df_raw = pd.read_csv(OUTPUT_PATH)
    category_counts = (
        df_raw.groupby(["material_name", df_raw["material_source"].fillna("")])
        ["impact_category"].nunique()
    )
    complete = set(category_counts[category_counts >= EXPECTED_N_CATEGORIES].index)
    incomplete = set(category_counts[category_counts < EXPECTED_N_CATEGORIES].index)

    stale = incomplete | {k for k in complete if k[0] in FORCE_REFRESH}
    df_existing = df_raw[~df_raw.apply(
        lambda r: (r["material_name"], r["material_source"] if pd.notna(r["material_source"]) else "") in stale,
        axis=1,
    )].copy()

    print(f"Found existing {OUTPUT_PATH} with {len(df_raw)} rows")
    print(f"  {len(complete)} activities complete (all {EXPECTED_N_CATEGORIES} categories present)")
    if incomplete:
        print(f"  ⚠ {len(incomplete)} activities INCOMPLETE — will re-download: {sorted(k[0] for k in incomplete)}")

    return df_existing, complete - set(FORCE_REFRESH)


# ── Row classification: what ecoinvent lookups does a sheet row need? ─────────
def lookup_keys_for_row(row):
    """(ecoinvent_name, location, ref_product, material_unit) keys a row needs —
    one key for a single-location row, one per country for a EU-wide row.
    Shared by estimate_runtime() and resolve_row() so they can't drift apart."""
    if pd.isna(row["ecoinventDataset_name"]):
        return []
    ref_product = row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None
    locations = EUROPEAN_LOCATIONS if row["geographicalCoverage"].lower() == "lookup: all european countries" else [row["geographicalCoverage"]]
    return [(row["ecoinventDataset_name"], loc, ref_product, row["material_unit"]) for loc in locations]

def source_or_none(row):
    """pd.NA-safe: material_source column is pandas 'string' dtype, so missing
    values are pd.NA — `x or default` raises TypeError on pd.NA, unlike np.nan."""
    return row["material_source"] if pd.notna(row["material_source"]) else None

def estimate_runtime(cached_keys):
    """Dry-run: count unique ecoinvent lookups still needed, after disk + dedup."""
    unique_keys = set()
    n_transport = n_biopol = 0

    for _, row in df_sources.iterrows():
        if (row["activity_name"], source_or_none(row) or "") in cached_keys:
                    continue
        if pd.notna(row["ecoinventDataset_name"]):
            unique_keys.update(lookup_keys_for_row(row))
        elif row["alternative_data_source"] == "road transport 50 km":
            n_transport += 1
        elif row["alternative_data_source"] == "biopol_lca_database":
            n_biopol += 1

    est_seconds = len(unique_keys) * LOOKUP_SECONDS_ESTIMATE
    print(f"\nEstimated ecoinvent lookups needed: {len(unique_keys)} (after disk + in-run dedup)")
    if n_transport:
        print(f"  + {n_transport} road-transport row(s) — reuse transport_act, negligible time")
    if n_biopol:
        print(f"  + {n_biopol} biopol_lca row(s) — reuse binder_scores, negligible time")
    print(f"Estimated runtime: ~{est_seconds:.0f} sec (~{est_seconds/60:.1f} min)\n")
    return len(unique_keys)


# ── In-memory ecoinvent lookup cache ───────────────────────────────────────────
ei_score_cache = {}
ei_cache_hits = ei_cache_misses = 0

def get_ei_scores(ecoinvent_name, location, ref_product, material_unit):
    """Scores (all EF v3.1 categories) for one ecoinvent activity, memoized by identity —
    reused across any sheet row pointing at the same (dataset, location, ref_product, unit)."""
    global ei_cache_hits, ei_cache_misses
    key = (ecoinvent_name, location, ref_product, material_unit)
    if key in ei_score_cache:
        ei_cache_hits += 1
        return ei_score_cache[key]
    ei_cache_misses += 1

    try:
        act = find_ei(ecoinvent_name, location, ref_product)
    except ValueError:
        ei_score_cache[key] = None
        return None

    kg_factor = get_kg_conversion_factor(act) if material_unit == "kg" else None
    if material_unit == "kg" and kg_factor is None:
        print(f"⚠ '{ecoinvent_name}' | {location}: no kg-conversion available — skipped")
        ei_score_cache[key] = None
        return None

    scores = {}
    for method in EF_METHODS:
        lca = bc.LCA({act: 1}, method)
        lca.lci()
        lca.lcia()
        scores[method[2]] = lca.score / kg_factor if kg_factor else lca.score
    ei_score_cache[key] = scores
    return scores


# ── Row builder — the one place the output dict shape is defined ──────────────
def make_rows(activity_name, material_unit, material_source, location, scores, lca_database, lca_method):
    """scores: either {impact_category: score} or a list of (impact_category, score, lca_database, lca_method) —
    normalized here so every branch below can call this the same way."""
    return [{
        "material_name": activity_name,
        "material_unit": material_unit,
        "location": location,
        "impact_category": cat,
        "score": score,
        "lca_database": lca_database,
        "lca_method": lca_method,
        "material_source": material_source,
    } for cat, score in scores.items()]


# ── Per-row resolution — one branch per data source type ──────────────────────
def resolve_ecoinvent_row(row):
    rows, errors = [], []
    for ecoinvent_name, location, ref_product, material_unit in lookup_keys_for_row(row):
        print(f"  {location} ...")
        scores = get_ei_scores(ecoinvent_name, location, ref_product, material_unit)
        if scores is None:
            errors.append((f"{row['activity_name']} | {location}", "resolution/conversion failed"))
            continue
        rows += make_rows(row["activity_name"], material_unit, source_or_none(row),
                           location, scores, EF_METHODS[0][0], EF_METHODS[0][1])
    return rows, errors


def resolve_transport_row(row):
    scores = {}
    for method in EF_METHODS:
        lca = bc.LCA({transport_act: 1}, method)
        lca.lci()
        lca.lcia()
        scores[method[2]] = lca.score * tkm_per_kg
    return make_rows(row["activity_name"], row["material_unit"], source_or_none(row),
                      row["geographicalCoverage"], scores, EF_METHODS[0][0], EF_METHODS[0][1]), []


def resolve_biopol_row(row):
    return make_rows(row["activity_name"], row["material_unit"], source_or_none(row),
                      row["geographicalCoverage"], binder_scores, "biopol_lca", "EF v3.1"), []


def resolve_row(row):
    """Dispatch a single df_sources row to the right resolver."""
    if pd.notna(row["ecoinventDataset_name"]):
        return resolve_ecoinvent_row(row)
    if row["alternative_data_source"] == "road transport 50 km":
        return resolve_transport_row(row)
    if row["alternative_data_source"] == "biopol_lca_database":
        return resolve_biopol_row(row)
    return [], [(f"{row['activity_name']} | {row['geographicalCoverage']}",
                 "No ecoinventDataset_name or recognized alternative_data_source")]


# ── Run ─────────────────────────────────────────────────────────────────────
df_existing, cached_keys = load_cache_state()
estimated_n_lookups = estimate_runtime(cached_keys)

burden_rows, resolution_errors, skipped = [], [], []

for _, row in df_sources.iterrows():
    cache_key = (row["activity_name"], source_or_none(row) or "")
    if cache_key in cached_keys:
        skipped.append(row["activity_name"])
        continue

    print(f"Processing '{row['activity_name']}' ({source_or_none(row) or '—'}, {row['geographicalCoverage'] or '—'}) ...")
    new_rows, errors = resolve_row(row)
    burden_rows += new_rows
    resolution_errors += errors

print(f"\nDownloaded {len(burden_rows)} new rows")
print(f"Ecoinvent lookup cache: {ei_cache_hits} hits, {ei_cache_misses} misses ({ei_cache_hits} redundant LCA runs avoided)")
if skipped:
    print(f"Skipped {len(skipped)} already-complete activities: {sorted(set(skipped))}")
if resolution_errors:
    print(f"\n⚠ {len(resolution_errors)} unresolved:")
    for name, err in resolution_errors:
        print(f"  {name}: {err}")

Found existing ../../data/processed/unit_burdens.csv with 1725 rows
  30 activities complete (all 25 categories present)

Estimated ecoinvent lookups needed: 41 (after disk + in-run dedup)
Estimated runtime: ~820 sec (~13.7 min)

Processing 'avoided burden - incineration, heat' (—, RER) ...
  RER ...
Processing 'avoided burden - incineration, electricity' (—, lookup: all european countries) ...
  CH ...
  AL ...
  AT ...
  BA ...
  BE ...
  BG ...
  BY ...
  CZ ...
  DE ...
  DK ...
  EE ...
  ES ...
  FI ...
  FR ...
  GB ...
  GI ...
  GR ...
  HR ...
  HU ...
  IE ...
  IS ...
  IT ...
  LT ...
  LU ...
  LV ...
  MD ...
  ME ...
  MK ...
  MT ...
  NL ...
  NO ...
  PL ...
  PT ...
  RO ...
  RS ...
  SE ...
  SI ...
  SK ...
  UA ...
  XK ...

Downloaded 1025 new rows
Ecoinvent lookup cache: 0 hits, 41 misses (0 redundant LCA runs avoided)
Skipped 30 already-complete activities: ['Bark chips', 'Cellulose reject fibers', 'Cement', 'Cotton fiber', 'Fiber glue (epoxy resin)', 'Hard w

#### Assemble and export unit_burdens.csv

In [18]:
# ── Re-read disk state fresh, right before writing — don't trust in-memory df_existing ──
if os.path.exists(OUTPUT_PATH):
    df_on_disk = pd.read_csv(OUTPUT_PATH)
else:
    df_on_disk = pd.DataFrame()

df_new = pd.DataFrame(burden_rows)
if not df_new.empty:
    df_new["impact_category_unit"] = df_new["impact_category"].map(EF_UNITS)

KEY_COLS = ["material_name", "material_source", "location", "impact_category"]

# combine, then drop duplicate keys keeping the NEWEST version (df_new rows win over
# stale disk rows for anything re-downloaded this run, e.g. FORCE_REFRESH or incomplete fixes)
df_combined = pd.concat([df_on_disk, df_new], ignore_index=True)
if not df_combined.empty:
    df_combined = df_combined.drop_duplicates(subset=KEY_COLS, keep="last")

df_unit_burdens = df_combined[[
    "material_name", "material_unit", "location",
    "impact_category", "impact_category_unit", "score",
    "lca_database", "lca_method", "material_source",
]]

n_before = len(df_on_disk)
n_after = len(df_unit_burdens)

# ── Safety check: refuse to silently shrink the dataset ───────────────────────
# a legitimate shrink only happens if FORCE_REFRESH or incomplete-category fixes
# removed some rows that then failed to fully re-download — flag it instead of overwriting blind
if n_before > 0 and n_after < n_before:
    print(f"⚠️  WARNING: row count would DROP from {n_before} to {n_after}.")
    print("This usually means some materials failed to fully re-download this run.")
    print("Writing to unit_burdens_UNSAFE.csv instead — inspect it before renaming.")
    df_unit_burdens.to_csv(f"{OUTPUT_DIR}/unit_burdens_UNSAFE.csv", index=False)
else:
    n_activities = df_unit_burdens["material_name"].nunique()
    n_categories = df_unit_burdens["impact_category"].nunique()
    print(f"Unit burdens: {n_activities} activities × {n_categories} categories = {n_after} rows "
          f"({n_before} on disk before this run → {n_after} after, {len(df_new)} new rows added this run)")

    df_unit_burdens.to_csv(OUTPUT_PATH, index=False)
    print(f"Exported to {OUTPUT_PATH}")

df_unit_burdens.head()

Unit burdens: 27 activities × 25 categories = 2750 rows (1725 on disk before this run → 2750 after, 1025 new rows added this run)
Exported to ../../data/processed/unit_burdens.csv


,material_name,material_unit,location,impact_category,impact_category_unit,score,lca_database,lca_method,material_source
0,Hemp fiber,kg,NaN,acidification,mol H+-eq,0.028951,ecoinvent-3.12,EF v3.1,virgin
1,Hemp fiber,kg,NaN,climate change,kg CO2-eq,0.814968,ecoinvent-3.12,EF v3.1,virgin
2,Hemp fiber,kg,NaN,climate change: biogenic,kg CO2-eq,0.018624,ecoinvent-3.12,EF v3.1,virgin
3,Hemp fiber,kg,NaN,climate change: fossil,kg CO2-eq,0.795829,ecoinvent-3.12,EF v3.1,virgin
4,Hemp fiber,kg,NaN,climate change: land use and land use change,kg CO2-eq,0.000514,ecoinvent-3.12,EF v3.1,virgin


## Unit benefit constants

Extract `carbon content` and `rotation period` values for biobased materials based on literature and ecoinvent. Because the benefits (carbon sequestration) depends on both product mass *and* lifespan, we are calculating the *constants* for calculating benefits, not the benefits themselves. These can only be calculated when we know what the product lifespan is, and this is only defined in `02_model.ipynb`.

**Important note**: the unit for carbon content is kg *carbon (C)* per kg (wet) material, not kg *CO2* per kg material. So when calculating carbon sequestration in the model (next notebook `02_model.ipynb`), we first need to multiply this carbon content value by the molecular weight ratio between CO2 and C, which is about 3.67.

#### Pea protein binder — derived carbon fraction

Pea protein binder isn't pure pea — it's derived from the LCI chain
(`peas → AEIEP pea protein isolate production → pea protein binder production`)
in `biopol_lca`, same cross-project pattern as the burden calculation. The pea
mass fraction scales the pure-pea carbon fraction down to a binder-level value.

In [21]:
def get_exchange_amount(node, producer_name_contains):
    """Amount of a technosphere exchange, matched by (partial) producer name."""
    matches = [e for e in node.technosphere() if producer_name_contains in e.input['name']]
    if not matches:
        raise ValueError(f"No exchange found containing '{producer_name_contains}'")
    return matches[0]['amount']

bd.projects.set_current("biopol_lca")
node_isolate = bd.get_node(name="AEIEP pea protein isolate production", database="lca_database_3DPrintedBiopol")
node_binder  = bd.get_node(name="pea protein binder production",        database="lca_database_3DPrintedBiopol")

kg_peas_per_kg_isolate   = get_exchange_amount(node_isolate, "protein pea")
kg_isolate_per_kg_binder = get_exchange_amount(node_binder, "AEIEP pea protein isolate production")
kg_peas_per_kg_binder    = kg_peas_per_kg_isolate * kg_isolate_per_kg_binder

bd.projects.set_current(PROJECT_NAME)  # switch back before continuing

CARBON_FRACTION_PEA = 0.45  # pure whole dried pea, IPCC Tier 1 / Lal (2004)
carbon_fraction_binder = CARBON_FRACTION_PEA * kg_peas_per_kg_binder

print(f"kg peas / kg binder:            {kg_peas_per_kg_binder:.4f}")
print(f"Derived binder carbon fraction: {carbon_fraction_binder:.4f}")

kg peas / kg binder:            0.5893
Derived binder carbon fraction: 0.2652


#### Load benefits data sources (from Google Sheets) and resolve to ecoinvent activities

In [ ]:
import pandas as pd

SHEET_ID = "1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4"
BENEFITS_GID = "1607154287"  # unit_benefits_dataSources tab
benefits_sheet_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={BENEFITS_GID}"

df_benefits = pd.read_csv(benefits_sheet_url, na_values=["n/a"])

# drop blank rows and the trailing "note:" rows at the bottom of the sheet —
# real data rows always have a `unit` value ("1 kg"), the note rows don't
df_benefits = df_benefits[df_benefits["unit"].notna()].copy()

for col in ["material_name", "ecoinventDataset_name", "geographicalCoverage", "referenceProduct"]:
    df_benefits[col] = df_benefits[col].astype("string").str.strip()

print(f"Loaded {len(df_benefits)} materials from '{benefits_sheet_url.split('gid=')[0]}...'")

# ── Resolve each material with an ecoinvent dataset ───────────────────────────
benefit_ei_activities = {}
benefit_resolution_errors = []

resolvable_benefits = df_benefits[df_benefits["ecoinventDataset_name"].notna()]
no_ecoinvent = df_benefits[df_benefits["ecoinventDataset_name"].isna()]

for _, row in resolvable_benefits.iterrows():
    print(f"Resolving benefit '{row['material_name']}' → '{row['ecoinventDataset_name']}' | {row['geographicalCoverage']} ... ")
    try:
        act = find_ei(
            name=row["ecoinventDataset_name"],
            location=row["geographicalCoverage"],
            ref_product=row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None,
        )
        benefit_ei_activities[row["material_name"]] = act
    except ValueError as e:
        benefit_resolution_errors.append((row["material_name"], str(e)))

print(f"Resolved:  {len(benefit_ei_activities)} / {len(resolvable_benefits)}")
print(f"No ecoinvent dataset (handled separately): {no_ecoinvent['material_name'].tolist()}")
if benefit_resolution_errors:
    print(f"\n⚠ {len(benefit_resolution_errors)} lookup(s) FAILED:")
    for name, err in benefit_resolution_errors:
        print(f"  {name}: {err}")

#### Extract carbon content data

In [22]:
import pandas as pd

# resolve biobased benefits to ecoinvent datasets, for later scoring

DATA_DIR = "../../data/data_sources"  # adjust to match your repo structure

# ── Reload the updated benefits CSV ───────────────────────────────────────────
df_benefits = pd.read_csv(f"{DATA_DIR}/materials_biobased_benefits.csv", na_values=["n/a"])
for col in ["material_name", "ecoinventDataset_name", "geographicalCoverage", "referenceProduct"]:
    df_benefits[col] = df_benefits[col].astype("string").str.strip()

# ── Resolve each material with an ecoinvent dataset ───────────────────────────
benefit_ei_activities = {}
benefit_resolution_errors = []

resolvable_benefits = df_benefits[df_benefits["ecoinventDataset_name"].notna()]
no_ecoinvent = df_benefits[df_benefits["ecoinventDataset_name"].isna()]

for _, row in resolvable_benefits.iterrows():
    print(f"Resolving benefit '{row['material_name']}' → '{row['ecoinventDataset_name']}' | {row['geographicalCoverage']} ... ")
    try:
        act = find_ei(
            name=row["ecoinventDataset_name"],
            location=row["geographicalCoverage"],
            ref_product=row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None,
        )
        benefit_ei_activities[row["material_name"]] = act
    except ValueError as e:
        benefit_resolution_errors.append((row["material_name"], str(e)))

print(f"Resolved:  {len(benefit_ei_activities)} / {len(resolvable_benefits)}")
print(f"No ecoinvent dataset (handled separately): {no_ecoinvent['material_name'].tolist()}")
if benefit_resolution_errors:
    print(f"\n⚠ {len(benefit_resolution_errors)} lookup(s) FAILED:")
    for name, err in benefit_resolution_errors:
        print(f"  {name}: {err}")

Resolving benefit 'Peas' → 'market for protein pea' | GLO ... 
Resolving benefit 'Cotton fiber' → 'market for seed-cotton' | GLO ... 
Resolving benefit 'Hemp fiber' → 'decorticated fibre production, hemp' | FR ... 
Resolving benefit 'Kenaf fiber' → 'market for fibre, kenaf' | GLO ... 
Resolving benefit 'Jute fiber' → 'market for fibre, jute' | GLO ... 
Resolving benefit 'Grass fiber' → 'market for grass fibre' | GLO ... 
Resolving benefit 'Bark chips' → 'market for bark chips, green, measured as dry mass' | Europe without Switzerland ... 
Resolving benefit 'Sawdust' → 'market for sawdust, green, collected, measured as dry mass' | Europe without Switzerland ... 
Resolving benefit 'Wood (hard wood, raw)' → 'market for sawnwood, hardwood, raw' | GLO ... 
Resolving benefit 'Wood (soft wood, raw)' → 'market for sawnwood, softwood, raw' | GLO ... 
Resolving benefit 'Cellulose reject fibers' → 'market for waste paper, sorted' | GLO ... 
Resolved:  11 / 11
No ecoinvent dataset (handled separat

In [23]:
# extrack raw properties 

def get_production_properties(act):
    """Return the properties dict from an activity's production exchange, or {} if none."""
    for exc in act.production():
        return exc.get("properties", {})
    return {}

material_properties = {}
for material_name, act in benefit_ei_activities.items():
    props = get_production_properties(act)
    material_properties[material_name] = props
    print(f"{material_name}  ({act['name']} | {act['location']})")
    if props:
        for key, val in props.items():
            print(f"    {key}: {val.get('amount')}")
    else:
        print("    ⚠ no properties found on production exchange")
    print()

Peas  (market for protein pea | GLO)
    carbon allocation: 0.407697116980715
    carbon content: 0.4694207145270348
    carbon content, fossil: 0.0
    carbon content, non-fossil: 0.4694207145270348
    dry mass: 0.8685111337523537
    energy content: 16.077599999999993
    price: 2.109999999999999
    water content: 0.15141521818690692
    water in wet mass: 0.13148886624764583
    wet mass: 0.9999999999999996

Cotton fiber  (market for seed-cotton | GLO)
    carbon allocation: 0.4384249999999996
    carbon content: 0.47499999999999976
    carbon content, fossil: 0.0
    carbon content, non-fossil: 0.47499999999999976
    dry mass: 0.9229999999999997
    price: 0.41101899999999986
    water content: 0.08342361863488616
    water in wet mass: 0.07699999999999996
    wet mass: 0.9999999999999996

Hemp fiber  (decorticated fibre production, hemp | FR)
    allocation factor: 0.606955220812199
    carbon allocation: 0.445704
    carbon content: 0.4548
    carbon content, fossil: 0.0
    c

### Compute carbon content per kg wet material

For every material with an ecoinvent activity, applies:

$$
\text{carbon content per kg wet material} = \text{carbon content, non-fossil} \times \frac{\text{dry mass}}{\text{wet mass}}
$$

Dividing by `wet mass` (rather than assuming it's 1) makes this correct regardless of the activity's actual reference unit.

**Seagrass** has no ecoinvent activity, so it's handled separately: its carbon content stays the literal literature value already in the sheet (0.336, dry-mass basis) — still not converted to a wet basis, an open gap.

In [ ]:
# ── Compute carbon content per kg wet material for all ecoinvent-resolved materials ──
carbon_content_rows = []

for material_name, props in material_properties.items():
    carbon_nonfossil = props.get("carbon content, non-fossil", {}).get("amount")
    dry_mass = props.get("dry mass", {}).get("amount")
    wet_mass = props.get("wet mass", {}).get("amount")

    if carbon_nonfossil is None or dry_mass is None or wet_mass is None:
        print(f"⚠ {material_name}: missing one of carbon content/dry mass/wet mass — skipped")
        continue

    carbon_per_kg_wet = carbon_nonfossil * (dry_mass / wet_mass)
    carbon_content_rows.append({
        "material_name": material_name,
        "carbon_content_kgC_per_kgWet": carbon_per_kg_wet,
        "carbon_content_source": "ecoinvent (carbon content, non-fossil × dry mass / wet mass)",
    })

carbon_content_rows.append({
    "material_name": "Pea protein binder",
    "carbon_content_kgC_per_kgWet": carbon_fraction_binder,
    "carbon_content_source": "derived from biopol_lca LCI chain (peas → isolate → binder), CARBON_FRACTION_PEA=0.45 (Lal, 2004)",
})

df_carbon_ecoinvent = pd.DataFrame(carbon_content_rows)

# ── Seagrass — literal literature value, no ecoinvent activity ───────────────
seagrass_row = df_benefits[df_benefits["material_name"] == "Seagrass"].iloc[0]
df_carbon_seagrass = pd.DataFrame([{
    "material_name": "Seagrass",
    "carbon_content_kgC_per_kgWet": seagrass_row["carbonContent_dryWeight_kg"],
    "carbon_content_source": seagrass_row["carbonContent_source"] + " [dry-mass basis, NOT converted to wet basis — open gap]",
}])

df_carbon_content = pd.concat([df_carbon_ecoinvent, df_carbon_seagrass], ignore_index=True)

print(f"Computed carbon content for {len(df_carbon_content)} materials:\n")
df_carbon_content

Computed carbon content for 13 materials:



,material_name,carbon_content_kgC_per_kg,carbon_content_source
0,Peas,0.407697,"ecoinvent (carbon content, non-fossil × dry ma..."
1,Cotton fiber,0.438425,"ecoinvent (carbon content, non-fossil × dry ma..."
2,Hemp fiber,0.445704,"ecoinvent (carbon content, non-fossil × dry ma..."
3,Kenaf fiber,0.418944,"ecoinvent (carbon content, non-fossil × dry ma..."
4,Jute fiber,0.404396,"ecoinvent (carbon content, non-fossil × dry ma..."
5,Grass fiber,0.45034,"ecoinvent (carbon content, non-fossil × dry ma..."
6,Bark chips,0.205833,"ecoinvent (carbon content, non-fossil × dry ma..."
7,Sawdust,0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."
8,"Wood (hard wood, raw)",0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."
9,"Wood (soft wood, raw)",0.290588,"ecoinvent (carbon content, non-fossil × dry ma..."


### save as csv

In [ ]:
OUTPUT_DIR = "../../data/processed"  # adjust to match your repo structure
OUTPUT_PATH = f"{OUTPUT_DIR}/benefits_constants.csv"

# ── Merge carbon content + rotation period + material_group ──────────────────
df_benefit_params = df_benefits[["material_name", "rotationPeriod_yr", "rotationPeriod_source"]].rename(
    columns={"rotationPeriod_yr": "rotation_period_yr", "rotationPeriod_source": "rotation_period_source"}
)

# ── Add pea protein binder as its own row before merging ─────────────────────
df_benefit_params_binder = pd.DataFrame([{
    "material_name": "Pea protein binder",
    "rotation_period_yr": 1,
    "rotation_period_source": "Annual crop, same as peas",
}])
df_benefit_params = pd.concat([df_benefit_params, df_benefit_params_binder], ignore_index=True)

df_benefit_params = df_benefit_params.merge(df_carbon_content, on="material_name", how="left")

df_benefit_params = df_benefit_params[[
    "material_name",
    "carbon_content_kgC_per_kgWet", "carbon_content_source",
    "rotation_period_yr", "rotation_period_source",
]]

# ── QA checks ─────────────────────────────────────────────────────────────────
missing_carbon = df_benefit_params[df_benefit_params["carbon_content_kgC_per_kgWet"].isna()]["material_name"].tolist()
if missing_carbon:
    print(f"⚠ {len(missing_carbon)} material(s) with no carbon content value: {missing_carbon}")

# ── Export ─────────────────────────────────────────────────────────────────────
df_benefit_params.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(df_benefit_params)} materials to {OUTPUT_PATH}")

df_benefit_params

All 12 biobased materials covered.

Written to ../../data/processed/unit_benefit_params.csv


,material_name,material_group,carbon_content_kgC_per_kg,carbon_content_source,rotation_period_yr,rotation_period_source
0,Peas,Food product,0.407697,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
1,Cotton fiber,Fiber,0.438425,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
2,Hemp fiber,Fiber,0.445704,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
3,Kenaf fiber,Fiber,0.418944,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
4,Jute fiber,Fiber,0.404396,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
5,Grass fiber,Fiber,0.45034,"ecoinvent (carbon content, non-fossil × dry ma...",1.0,Cherubini et al. 2011 (GWPbio methodology)
6,Bark chips,Forestry by-product,0.205833,"ecoinvent (carbon content, non-fossil × dry ma...",80.0,average between soft and hard wood
7,Sawdust,Forestry by-product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",80.0,average between soft and hard wood
8,"Wood (hard wood, raw)",Forestry product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",100.0,"Beech rotation (110–140 yr, or 80–100 yr in so..."
9,"Wood (soft wood, raw)",Forestry product,0.290588,"ecoinvent (carbon content, non-fossil × dry ma...",65.0,Softwood CORRIM figure: Cradle-to-gate LCA of ...


## End-of-life constants

Constants used in the model's Circular Footprint Formula (CFF) calculation at end-of-life, per material:

- **`allocation_factor`** — the CFF's *A* parameter. It splits the burdens/benefits of recycling between the material's current life and its next one. A high allocation factor (e.g. A = 0.8) means a low recycling rate, i.e. less-established recycling infrastructure for that material. The JRC allows three standard values: 0.2, 0.5, or 0.8.
- **`quality_ratio`** — the quality of the recycled material relative to virgin material (the CFF's *Qsin/Qp* ratio), which discounts the recycling credit when recycled material is lower-grade than virgin.
- **`lower_heating_value`** — how much usable energy is released when the material is incinerated (MJ/kg, dry basis). This drives the avoided-burden credit at end-of-life: incinerating the material displaces energy that would otherwise have been produced some other way (e.g. from natural gas), so a higher LHV means a bigger avoided burden.

These are combined with process burdens (composting, recycling, incineration, landfilling — from `unit_burdens.csv`) in `02_model.ipynb` to compute each material's net end-of-life impact.

In [ ]:
import pandas as pd

SHEET_ID = "1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4"
EOL_GID = "244012056"  # EoL_constants tab
eol_sheet_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={EOL_GID}"

df_eol = pd.read_csv(eol_sheet_url, na_values=["n/a"])

# drop blank rows and the trailing source/note rows at the bottom of the sheet —
# real data rows always have a `unit` value ("1 kg"), the note rows don't
df_eol = df_eol[df_eol["unit"].notna()].copy()

df_eol = df_eol.rename(columns={
    "material": "material_name",
    "lower_heating_value (MJ/kg, dry)": "lower_heating_value_MJ_per_kg_dry",
})

df_eol["material_name"] = df_eol["material_name"].astype("string").str.strip()
df_eol = df_eol[[
    "material_name", "allocation_factor", "quality_ratio",
    "lower_heating_value_MJ_per_kg_dry", "LHV_basis",
]]

print(f"Loaded {len(df_eol)} materials from '{eol_sheet_url.split('gid=')[0]}...'")
df_eol

### Save as csv

In [ ]:
OUTPUT_DIR = "../../data/processed"  # adjust to match your repo structure
OUTPUT_PATH = f"{OUTPUT_DIR}/eol_constants.csv"

df_eol.to_csv(OUTPUT_PATH, index=False)
print(f"Exported {len(df_eol)} materials to {OUTPUT_PATH}")